## Import

In [10]:
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [2]:
from datasets.Student_t import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.46
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## Joint training

In [12]:
## Vector copula estimation
from estimators.VCE_joint import VCE

hyperparams.K_components = 5
hyperparams.nde_type = 'VGC'

ret = []
time_spend = []
for i in range(8):
    t0 = time.time()
    
    estimator = VCE(None, None, architecture_critic, hyperparams)
    estimator.to(device)
    estimator.learn(X, Y)

    mi = estimator.MI(X, Y)
    t = time.time()-t0

    print('true MI:', dataset.mutual_information())
    print('est MI:', estimator.MI(X, Y))
    
    ret.append(mi)
    time_spend.append(t)

K components 5 joint learning True 

finished: t= 0 loss= 475.0346374511719 loss val= 487.22222900390625 best val loss= 487.22222900390625 best t= 0
finished: t= 101 loss= -165.21182250976562 loss val= -129.81512451171875 best val loss= -139.9361114501953 best t= 37


true MI: 12.456365411743121
est MI: 6.285105228424072
K components 5 joint learning True 

finished: t= 0 loss= 443.74920654296875 loss val= 448.5999450683594 best val loss= 448.5999450683594 best t= 0
finished: t= 101 loss= -169.5718231201172 loss val= -108.57485961914062 best val loss= -142.23709106445312 best t= 44


true MI: 12.456365411743121
est MI: 6.5943779945373535
K components 5 joint learning True 

finished: t= 0 loss= 495.728271484375 loss val= 534.4346313476562 best val loss= 534.4346313476562 best t= 0
finished: t= 101 loss= -174.2697296142578 loss val= -135.6605224609375 best val loss= -146.82022094726562 best t= 83
finished: t= 202 loss= -169.00027465820312 loss val= -139.4048614501953 best val loss= -146

In [14]:
ret = sorted(ret)

print(np.array(ret)[len(ret)//2], np.array(ret).std(), np.array(ret).max(), np.array(ret).min())
print(np.array(time_spend).mean(), np.array(time_spend).std())

7.818411350250244 5.550300944872234 10.118672370910645 -8.190896987915039
268.31857988238335 99.61939385914036
